### Лабораторная работа №2. Оценивание параметров распределения. Проверка статистических гипотез.

In [2]:
import pandas as pd
df = pd.read_csv('var_6.csv')

In [3]:
df

,v061,v062,v063,v064
0,6.407667,2.0,2.175204,1.518431
1,4.004938,5.0,1.474466,0.819543
2,7.268616,5.0,2.270366,-0.015757
3,4.653771,5.0,3.244601,0.931541
4,-1.198180,9.0,2.659261,2.143425
...,...,...,...,...
195,-4.442822,7.0,3.293761,-1.382387
196,4.362078,9.0,1.345542,-0.706161
197,0.995735,12.0,1.160786,-0.657356
198,9.466093,11.0,2.527722,0.660969


v061 (X) - нормальное распределение

v064 (Y) - нормальное распределение

v062 (D) - распределение Пуассона

v063 (Z) - равномерное распределение

In [4]:
#Вычислим оценки мм: 
from math import sqrt


print('Оценки ММ для v061 (X): ', round(df['v061'].mean(), 4), round(df['v061'].var(ddof=0), 4))
print('Оценки ММ для v064 (Y): ', round(df['v064'].mean(), 4), round(df['v064'].var(ddof=0), 4))
print('Оценки ММ для v062 (D): ', round(df['v062'].mean(), 4))
print('Оценки ММ для v063 (Z): ', round(df['v063'].mean() - sqrt(3)*df['v063'].std(ddof=0), 4), round(df['v063'].mean() + sqrt(3)*df['v063'].std(ddof=0), 4))

Оценки ММ для v061 (X):  2.6934 9.1274
Оценки ММ для v064 (Y):  0.9032 0.9508
Оценки ММ для v062 (D):  9.185
Оценки ММ для v063 (Z):  0.0753 3.878


In [5]:
#Вычислим оценки ммп, первые 3 совпадают:
print(f'v061: {round(df['v061'].mean(), 4)}, {round(df['v061'].var(ddof=0), 4)}')
print(f'v064: {round(df['v064'].mean(), 4)}, {round(df['v064'].var(ddof=0), 4)}')
print(f'v062: {round(df['v062'].mean(), 4)}')
print(f'v063: {min(df['v063'])}, {max(df['v063'])}')

v061: 2.6934, 9.1274
v064: 0.9032, 0.9508
v062: 9.185
v063: 0.054543, 3.99009


### Задание 2

In [25]:
# построим доверительный интервал для v061 (X) и v064 (Y) при уровне значимости 0.05 (для среднего)
import scipy.stats as stats 
import numpy as np


def confidence_interval_for_mean(sample):
    sample_mean = df[sample].mean()
    sample_std = df[sample].std(ddof=1)
    t_krit = stats.t.ppf(1-0.05/2, len(df[sample])-1)
    return sample_mean - t_krit*sample_std/np.sqrt(len(df[sample])), sample_mean + t_krit*sample_std/np.sqrt(len(df[sample]))


print(f'Доверительный интервал для v061 (X): [{round(confidence_interval_for_mean("v061")[0], 4)}, {round(confidence_interval_for_mean("v061")[1], 4)}]')
print(f'Доверительный интервал для v064 (Y): [{round(confidence_interval_for_mean("v064")[0], 4)}, {round(confidence_interval_for_mean("v064")[1], 4)}]')


Доверительный интервал для v061 (X): [2.2711, 3.1157]
Доверительный интервал для v064 (Y): [0.7669, 1.0395]


In [26]:
# построим доверительный интервал для v061 (X) и v064 (Y) при уровне значимости 0.05 (для дисперсии)

def confidence_interval_for_variance(sample):
    sample_var = df[sample].var()
    n = len(df[sample])
    H1 = stats.chi2.ppf(1-0.05/2, n-1)
    H2 = stats.chi2.ppf(0.05/2, n-1)
    return (n-1)*sample_var/H1, (n-1)*sample_var/H2
print(f'Доверительный интервал для v061 (X): [{round(confidence_interval_for_variance("v061")[0], 4)}, {round(confidence_interval_for_variance("v061")[1], 4)}]')

print(f'Доверительный интервал для v064 (Y): [{round(confidence_interval_for_variance("v064")[0], 4)}, {round(confidence_interval_for_variance("v064")[1], 4)}]')


Доверительный интервал для v061 (X): [7.6074, 11.2805]
Доверительный интервал для v064 (Y): [0.7925, 1.1751]


In [27]:
# построим доверительный интервал для отношения дисперсий v061 (X) и v064 (Y) при уровне значимости 0.05

def confidence_interval_for_variance_ratio(sample1, sample2):
    var1 = df[sample1].var()
    var2 = df[sample2].var()
    n1 = len(df[sample1])
    n2 = len(df[sample2])
    F1 = stats.f.ppf(1-0.05/2, n1-1, n2-1)
    F2 = stats.f.ppf(0.05/2, n1-1, n2-1)
    return (var1/var2)/F1, (var1/var2)/F2

print(f'Доверительный интервал для отношения дисперсий v061 (X) и v064 (Y): [{round(confidence_interval_for_variance_ratio("v061", "v064")[0], 4)}, {round(confidence_interval_for_variance_ratio("v061", "v064")[1], 4)}]')

Доверительный интервал для отношения дисперсий v061 (X) и v064 (Y): [7.2648, 12.6845]


In [31]:
# доверительный интервал для разности средних v061 (X) и v064 (Y) при уровне значимости 0.05

def confidence_interval_for_mean_difference(sample1, sample2):
    mean1 = df[sample1].mean()
    mean2 = df[sample2].mean()
    var1 = df[sample1].var()
    var2 = df[sample2].var()
    n1 = len(df[sample1])
    n2 = len(df[sample2])
    A = var1 / n1
    B = var2 / n2
    nu = (A + B)**2 / (A**2 / (n1 - 1) + B**2 / (n2 - 1))
    t_krit = stats.t.ppf(0.975, df=nu)
    se = np.sqrt((var1/n1) + (var2/n2))
    return (mean1 - mean2) - t_krit*se, (mean1 - mean2) + t_krit*se

print(f'Доверительный интервал для разности средних v061 (X) и v064 (Y): [{round(confidence_interval_for_mean_difference("v061", "v064")[0], 4)}, {round(confidence_interval_for_mean_difference("v061", "v064")[1], 4)}]')

Доверительный интервал для разности средних v061 (X) и v064 (Y): [1.3469, 2.2335]


In [35]:
# проверка гипотезы о равенстве средних v061 (X) и v064 (Y) при уровне значимости 0.05
result = stats.ttest_ind(df['v061'], df['v064'], equal_var=False)
print(f'Статистика теста: {round(result.statistic, 4)}, p-value: {round(result.pvalue, 14)}')

Статистика теста: 7.9549, p-value: 7e-14


In [36]:
# проверим гипотезу об отношении дисперсий v061 (X) и v064 (Y) при уровне значимости 0.05
x = df['v061']
y = df['v064']

f_stat = x.var(ddof=1) / y.var(ddof=1)
df1 = len(x) - 1
df2 = len(y) - 1

p_value = min(
    1.0,
    2 * min(
        stats.f.cdf(f_stat, df1, df2),
        stats.f.sf(f_stat, df1, df2)
    )
)

print(f'F = {f_stat:.4f}, p-value = {p_value:.4e}')
print('H0 отвергается' if p_value < 0.05 else 'H0 не отвергается')

F = 9.5995, p-value = 2.8098e-48
H0 отвергается


### Задание 3.

In [41]:
# тест манна-уитни для проверки гипотезы о равенстве распределений v061 (X) и v064 (Y) при уровне значимости 0.05
result = stats.mannwhitneyu(df['v061'], df['v064'], alternative='two-sided')
print(f'Статистика теста: {result.statistic}, p-value: {result.pvalue:.4e}')

Статистика теста: 29157.0, p-value: 2.3776e-15


In [42]:
# критерий знаков
differences = df['v061'] - df['v064']
k = (differences > 0).sum()
m = (differences != 0).sum()

result_sign = stats.binomtest(k, n=m, p=0.5, alternative='two-sided')

print(f'Положительных разностей: {k} из {m}')
print(f'p-value: {result_sign.pvalue:.4e}')

Положительных разностей: 146 из 200
p-value: 5.6743e-11


In [ ]:
# тест вилкоксона для проверки гипотезы о нулевом центре симметрии разностей v061 (X) и v064 (Y) при уровне значимости 0.05
result_w = stats.wilcoxon(
    df['v061'],
    df['v064'],
    alternative='two-sided'
)
print(f'Статистика теста: {result_w.statistic}, p-value: {result_w.pvalue:.4e}')    

Статистика теста: 4255.0, p-value: 1.5395e-12


In [45]:
# критерий хи-квадрат Пирсона

def normal_chi2_stat(sample, k=10):
    """Статистика Пирсона с k равновероятными интервалами."""
    sample = np.asarray(sample)
    n = len(sample)

    mu = sample.mean()
    sigma = sample.std(ddof=0)

    edges = stats.norm.ppf(
        np.linspace(0, 1, k + 1),
        loc=mu,
        scale=sigma
    )

    observed, _ = np.histogram(sample, bins=edges)
    expected = np.full(k, n / k)

    chi2_stat = np.sum((observed - expected)**2 / expected)
    return chi2_stat, observed, expected


def pearson_normal_test(sample, k=10, simulations=9999, seed=42):
    """Проверка нормальности с моделированием нулевого распределения."""
    sample = np.asarray(sample)
    n = len(sample)

    statistic, observed, expected = normal_chi2_stat(sample, k)
    rng = np.random.default_rng(seed)

    simulated_stats = np.empty(simulations)

    for i in range(simulations):
        # Достаточно N(0, 1): статистика не меняется
        # при сдвиге и положительном масштабировании данных.
        simulated_sample = rng.normal(size=n)
        simulated_stats[i] = normal_chi2_stat(simulated_sample, k)[0]

    p_value = (
        1 + np.count_nonzero(simulated_stats >= statistic)
    ) / (simulations + 1)

    return statistic, p_value, observed, expected


alpha = 0.05

for column in ['v061', 'v064']:
    statistic, p_value, observed, expected = pearson_normal_test(df[column])

    print(f'\nВыборка {column}')
    print(pd.DataFrame({
        'Интервал': np.arange(1, len(observed) + 1),
        'Наблюдаемая частота': observed,
        'Ожидаемая частота': expected
    }).to_string(index=False))

    print(f'Статистика Пирсона: {statistic:.4f}')
    print(f'p-value (моделирование): {p_value:.4f}')
    print(
        'Гипотеза нормальности отвергается'
        if p_value < alpha
        else 'Нет оснований отвергать гипотезу нормальности'
    )


Выборка v061
 Интервал  Наблюдаемая частота  Ожидаемая частота
        1                   19               20.0
        2                   14               20.0
        3                   25               20.0
        4                   20               20.0
        5                   24               20.0
        6                   25               20.0
        7                   17               20.0
        8                   19               20.0
        9                   14               20.0
       10                   23               20.0
Статистика Пирсона: 7.9000
p-value (моделирование): 0.3751
Нет оснований отвергать гипотезу нормальности

Выборка v064
 Интервал  Наблюдаемая частота  Ожидаемая частота
        1                   19               20.0
        2                   23               20.0
        3                   22               20.0
        4                   15               20.0
        5                   23               20.0
        6        

In [46]:
def uniform_chi2_stat(sample, k=10):
    sample = np.asarray(sample)
    n = len(sample)

    a = sample.min()
    b = sample.max()

    # Для равномерного закона равные по ширине интервалы
    # имеют одинаковые вероятности.
    edges = np.linspace(a, b, k + 1)

    observed, _ = np.histogram(sample, bins=edges)
    expected = np.full(k, n / k)

    statistic = np.sum((observed - expected)**2 / expected)
    return statistic, observed, expected


def pearson_uniform_test(sample, k=10, simulations=9999, seed=42):
    sample = np.asarray(sample)
    n = len(sample)

    statistic, observed, expected = uniform_chi2_stat(sample, k)
    rng = np.random.default_rng(seed)

    simulated_stats = np.empty(simulations)

    for i in range(simulations):
        # Достаточно U(0, 1): сдвиг и масштаб не меняют статистику.
        simulated_sample = rng.uniform(size=n)
        simulated_stats[i] = uniform_chi2_stat(simulated_sample, k)[0]

    p_value = (
        1 + np.count_nonzero(simulated_stats >= statistic)
    ) / (simulations + 1)

    return statistic, p_value, observed, expected


statistic, p_value, observed, expected = pearson_uniform_test(df['v063'])

print(pd.DataFrame({
    'Интервал': np.arange(1, len(observed) + 1),
    'Наблюдаемая частота': observed,
    'Ожидаемая частота': expected
}).to_string(index=False))

print(f'Статистика Пирсона: {statistic:.4f}')
print(f'p-value (моделирование): {p_value:.4f}')
print(
    'Гипотеза равномерного распределения отвергается'
    if p_value < 0.05
    else 'Нет оснований отвергать гипотезу равномерного распределения'
)

 Интервал  Наблюдаемая частота  Ожидаемая частота
        1                   18               20.0
        2                   22               20.0
        3                   16               20.0
        4                   28               20.0
        5                   19               20.0
        6                   24               20.0
        7                   20               20.0
        8                   18               20.0
        9                   17               20.0
       10                   18               20.0
Статистика Пирсона: 6.1000
p-value (моделирование): 0.7306
Нет оснований отвергать гипотезу равномерного распределения


In [47]:
def poisson_chi2_stat(sample):
    sample = np.asarray(sample)
    n = len(sample)
    lam = sample.mean()

    if lam == 0:
        return 0.0, np.array([n]), np.array([float(n)])

    # Границы групп около децилей распределения.
    # Убираем совпадающие квантили.
    cuts = np.unique(
        stats.poisson.ppf(np.arange(1, 10) / 10, mu=lam)
    )

    # +0.5 размещает границы между целыми значениями.
    # Крайние группы включают хвосты распределения.
    edges = np.r_[-np.inf, cuts + 0.5, np.inf]

    observed, _ = np.histogram(sample, bins=edges)
    probabilities = np.diff(stats.poisson.cdf(edges, mu=lam))
    expected = n * probabilities

    # Объединяем группы с малой ожидаемой частотой.
    while len(expected) > 1 and np.any(expected < 5):
        i = np.flatnonzero(expected < 5)[0]
        j = i + 1 if i < len(expected) - 1 else i - 1

        observed[j] += observed[i]
        expected[j] += expected[i]

        observed = np.delete(observed, i)
        expected = np.delete(expected, i)

    statistic = np.sum((observed - expected)**2 / expected)
    return statistic, observed, expected


def pearson_poisson_test(sample, simulations=9999, seed=42):
    sample = np.asarray(sample)
    n = len(sample)
    lam = sample.mean()

    statistic, observed, expected = poisson_chi2_stat(sample)
    rng = np.random.default_rng(seed)

    simulated_stats = np.empty(simulations)

    for i in range(simulations):
        simulated_sample = rng.poisson(lam=lam, size=n)

        # Внутри функции заново оцениваем lambda,
        # строим группы и вычисляем ожидаемые частоты.
        simulated_stats[i] = poisson_chi2_stat(simulated_sample)[0]

    p_value = (
        1 + np.count_nonzero(simulated_stats >= statistic)
    ) / (simulations + 1)

    return statistic, p_value, observed, expected


statistic, p_value, observed, expected = pearson_poisson_test(df['v062'])

print(pd.DataFrame({
    'Группа': np.arange(1, len(observed) + 1),
    'Наблюдаемая частота': observed,
    'Ожидаемая частота': expected
}).round(4).to_string(index=False))

print(f'Статистика Пирсона: {statistic:.4f}')
print(f'p-value (моделирование): {p_value:.4f}')
print(
    'Гипотеза распределения Пуассона отвергается'
    if p_value < 0.05
    else 'Нет оснований отвергать гипотезу распределения Пуассона'
)

 Группа  Наблюдаемая частота  Ожидаемая частота
      1                   18            20.9818
      2                   39            39.5542
      3                   26            25.7720
      4                   28            26.3017
      5                   29            24.1582
      6                   22            20.1721
      7                   14            15.4400
      8                   12            10.9090
      9                   12            16.7110
Статистика Пирсона: 3.2508
p-value (моделирование): 0.8664
Нет оснований отвергать гипотезу распределения Пуассона


In [48]:
def fitted_ks_stat(sample, distribution):
    sample = np.sort(np.asarray(sample))
    n = len(sample)

    if distribution == 'poisson':
        lam = sample.mean()

        # Проверяем расстояния до и после скачков обеих CDF.
        values, counts = np.unique(sample, return_counts=True)
        empirical_right = np.cumsum(counts) / n
        empirical_left = empirical_right - counts / n

        theoretical_right = stats.poisson.cdf(values, mu=lam)
        theoretical_left = stats.poisson.cdf(values - 1, mu=lam)

        return max(
            np.max(np.abs(empirical_right - theoretical_right)),
            np.max(np.abs(empirical_left - theoretical_left))
        )

    if distribution == 'norm':
        theoretical = stats.norm.cdf(
            sample,
            loc=sample.mean(),
            scale=sample.std(ddof=0)
        )
    elif distribution == 'uniform':
        theoretical = stats.uniform.cdf(
            sample,
            loc=sample.min(),
            scale=sample.max() - sample.min()
        )
    else:
        raise ValueError('Неизвестное распределение')

    empirical_right = np.arange(1, n + 1) / n
    empirical_left = np.arange(n) / n

    return max(
        np.max(empirical_right - theoretical),
        np.max(theoretical - empirical_left)
    )


def fitted_ks_test(sample, distribution, simulations=9999, seed=42):
    sample = np.asarray(sample)
    n = len(sample)
    rng = np.random.default_rng(seed)

    statistic = fitted_ks_stat(sample, distribution)
    simulated_stats = np.empty(simulations)

    for i in range(simulations):
        if distribution == 'norm':
            simulated_sample = rng.normal(size=n)
        elif distribution == 'uniform':
            simulated_sample = rng.uniform(size=n)
        elif distribution == 'poisson':
            simulated_sample = rng.poisson(sample.mean(), size=n)
        else:
            raise ValueError('Неизвестное распределение')

        # Параметры заново оцениваются внутри fitted_ks_stat.
        simulated_stats[i] = fitted_ks_stat(
            simulated_sample, distribution
        )

    p_value = (
        1 + np.count_nonzero(simulated_stats >= statistic)
    ) / (simulations + 1)

    return statistic, p_value


models = {
    'v061': 'norm',
    'v064': 'norm',
    'v062': 'poisson',
    'v063': 'uniform'
}

results_ks = []

for column, distribution in models.items():
    statistic, p_value = fitted_ks_test(df[column], distribution)

    results_ks.append({
        'Выборка': column,
        'Распределение': distribution,
        'D': statistic,
        'p-value': p_value,
        'Решение': (
            'H0 отвергается'
            if p_value < 0.05
            else 'H0 не отвергается'
        )
    })

print(pd.DataFrame(results_ks).round(4).to_string(index=False))

Выборка Распределение      D  p-value           Решение
   v061          norm 0.0525   0.1904 H0 не отвергается
   v064          norm 0.0301   0.9409 H0 не отвергается
   v062       poisson 0.0253   0.7842 H0 не отвергается
   v063       uniform 0.0474   0.7415 H0 не отвергается
